In [1]:
import os

BASE_PATH = r"clas\CLAS_Database\CLAS\Participants"

train_subjects = range(1, 43)


In [40]:
def get_training_files(subject_id):

    subject_folder = os.path.join(
        BASE_PATH, 
        f"part{subject_id}", 
        "by_block"
    )

    selected_files = []

    if not os.path.exists(subject_folder):
        print(f" Folder saknas för part{subject_id}")
        return selected_files

    for file in os.listdir(subject_folder):

        file_lower = file.lower()

        # Måste vara gsr_ppg csv
        if "gsr_ppg" not in file_lower:
            continue

        if not file_lower.endswith(".csv"):
            continue

        # Ta bort meta_info
        if "meta_info" in file_lower:
            continue

        # Ta bort start och end (filnummer 0 och 38)
        if file_lower.startswith("0_"):
            continue

        if file_lower.startswith("38_"):
            continue

        # Ta bort ECG om det finns
        if "ecg" in file_lower:
            continue

        selected_files.append(os.path.join(subject_folder, file))

    return selected_files


In [41]:
train_files = []

for subject in train_subjects:
    files = get_training_files(subject)
    train_files.extend(files)

print(f"Total train files : {len(train_files)}")


Total train files : 1554


In [42]:
import pandas as pd

test_df = pd.read_csv(train_files[0])
print(test_df.head())


   Timestamp  PythonTimestamp  accelx  accely  accelz          ppg        gsr
0     307383     1.551793e+09    2041    2825    2145  1372.161172  66.840983
1     307511     1.551793e+09    2040    2823    2143  1373.626374  66.840983
2     307639     1.551793e+09    2041    2823    2141  1375.091575  66.795396
3     307767     1.551793e+09    2042    2821    2139  1374.358974  66.840983
4     307895     1.551793e+09    2040    2822    2139  1372.893773  66.795396


In [69]:
import os
import numpy as np
import pandas as pd
import neurokit2 as nk
from tqdm import tqdm

FS = 256
WINDOW_SEC = 30
STEP_SEC = 15

window_size = WINDOW_SEC * FS
step_size = STEP_SEC * FS


In [88]:
test_file = train_files[0]
df = pd.read_csv(test_file)

print(df.head())
print("Längd:", len(df))


   Timestamp  PythonTimestamp  accelx  accely  accelz          ppg        gsr
0     307383     1.551793e+09    2041    2825    2145  1372.161172  66.840983
1     307511     1.551793e+09    2040    2823    2143  1373.626374  66.840983
2     307639     1.551793e+09    2041    2823    2141  1375.091575  66.795396
3     307767     1.551793e+09    2042    2821    2139  1374.358974  66.840983
4     307895     1.551793e+09    2040    2822    2139  1372.893773  66.795396
Längd: 7678


In [72]:
acc_mag = np.sqrt(
    df["accelx"].values**2 +
    df["accely"].values**2 +
    df["accelz"].values**2
)

print("ACC mean:", np.mean(acc_mag))
print("ACC std:", np.std(acc_mag))


ACC mean: 2932.730274361709
ACC std: 3.3832571900083868


In [90]:
df = pd.read_csv(valid_files[0])

eda_raw = df["gsr"].values
ppg_raw = df["ppg"].values

eda_clean = nk.eda_clean(eda_raw, sampling_rate=FS)
ppg_clean = nk.ppg_clean(ppg_raw, sampling_rate=FS)

print("Signal length:", len(eda_clean))
print("EDA NaN:", np.isnan(eda_clean).sum())
print("PPG NaN:", np.isnan(ppg_clean).sum())


Signal length: 15367
EDA NaN: 0
PPG NaN: 0


In [91]:
WINDOW_SEC = 30
STEP_SEC = 15

window_size = WINDOW_SEC * FS
step_size = STEP_SEC * FS

windows = []

for start in range(0, len(eda_clean) - window_size + 1, step_size):
    end = start + window_size
    
    window = {
        "ppg": ppg_clean[start:end],
        "gsr": eda_clean[start:end],
        "acc": acc_mag[start:end]
    }
    
    windows.append(window)

print("Antal windows:", len(windows))
print("Window length (samples):", len(windows[0]["ppg"]) if windows else 0)


Antal windows: 3
Window length (samples): 7680


In [92]:
def filter_acc_windows(windows):
    
    motion_levels = np.array([np.std(w["acc"]) for w in windows])
    
    Q1 = np.percentile(motion_levels, 25)
    Q3 = np.percentile(motion_levels, 75)
    IQR = Q3 - Q1
    
    threshold = Q3 + 1.5 * IQR
    
    keep_mask = motion_levels <= threshold
    
    filtered = [windows[i] for i in range(len(windows)) if keep_mask[i]]
    
    print("Motion levels:", motion_levels)
    print("Threshold:", threshold)
    print(f"Behållna windows: {len(filtered)} / {len(windows)}")
    
    return filtered

windows_filtered = filter_acc_windows(windows)


Motion levels: [3.11911359 3.09660894 2.53401011]
Threshold: 3.546688879590386
Behållna windows: 3 / 3


In [93]:
def extract_hrv_features(ppg_window):
    
    signals, info = nk.ppg_process(ppg_window, sampling_rate=FS)
    peaks = info["PPG_Peaks"]

    print("Antal peaks:", len(peaks))

    if len(peaks) < 20:
        print("För få peaks → skip")
        return None

    hrv = nk.hrv_time(peaks, sampling_rate=FS)

    return {
        "MeanNN_ms": hrv["HRV_MeanNN"].values[0],
        "SDNN_ms": hrv["HRV_SDNN"].values[0],
        "RMSSD_ms": hrv["HRV_RMSSD"].values[0]
    }

hrv_test = extract_hrv_features(windows_filtered[0]["ppg"])
print("HRV:", hrv_test)


Antal peaks: 38
HRV: {'MeanNN_ms': np.float64(777.9771959459459), 'SDNN_ms': np.float64(120.08679646557756), 'RMSSD_ms': np.float64(164.14256892061132)}


In [94]:
def extract_eda_features(gsr_window):
    
    eda_signals, info = nk.eda_process(gsr_window, sampling_rate=FS)
    eda_feat = nk.eda_analyze(eda_signals, sampling_rate=FS)

    scr_n = eda_feat["SCR_Peaks_N"].values[0]
    scr_amp = eda_feat["SCR_Peaks_Amplitude_Mean"].values[0]
    tonic_sd = eda_feat["EDA_Tonic_SD"].values[0]
    auto = eda_feat["EDA_Autocorrelation"].values[0]

    print("SCR count:", scr_n)
    print("SCR amplitude:", scr_amp)
    print("Tonic SD:", tonic_sd)
    print("Autocorrelation:", auto)

    return {
        "SCR_Peaks_N": scr_n,
        "SCR_Amplitude_Mean": scr_amp,
        "EDA_Tonic_SD": tonic_sd,
        "EDA_Autocorrelation": auto
    }

eda_test = extract_eda_features(windows_filtered[0]["gsr"])
print("EDA:", eda_test)


SCR count: 1.0
SCR amplitude: 4.553109028058762
Tonic SD: 4.108623775632601
Autocorrelation: nan
EDA: {'SCR_Peaks_N': np.float64(1.0), 'SCR_Amplitude_Mean': np.float64(4.553109028058762), 'EDA_Tonic_SD': np.float64(4.108623775632601), 'EDA_Autocorrelation': np.float64(nan)}


In [101]:
feature_rows = []

WINDOW_SEC = 30
STEP_SEC = 15

window_size = WINDOW_SEC * FS
step_size = STEP_SEC * FS

for file in tqdm(valid_files, desc="Processing files"):

    try:
        df = pd.read_csv(file)

        # Hoppa över filer < 30 sek
        if len(df) < window_size:
            continue

        participant_id = int(file.split("part")[1].split("\\")[0])

        # ---------- CLEAN ----------
        eda_clean = nk.eda_clean(df["gsr"].values, sampling_rate=FS)
        ppg_clean = nk.ppg_clean(df["ppg"].values, sampling_rate=FS)

        acc_mag = np.sqrt(
            df["accelx"].values**2 +
            df["accely"].values**2 +
            df["accelz"].values**2
        )

        # ---------- WINDOWING ----------
        windows = []

        for start in range(0, len(eda_clean) - window_size + 1, step_size):
            end = start + window_size

            windows.append({
                "ppg": ppg_clean[start:end],
                "gsr": eda_clean[start:end],
                "acc": acc_mag[start:end]
            })

        if len(windows) == 0:
            continue

        # ---------- ACC FILTER ----------
        motion_levels = np.array([np.std(w["acc"]) for w in windows])

        Q1 = np.percentile(motion_levels, 25)
        Q3 = np.percentile(motion_levels, 75)
        IQR = Q3 - Q1
        threshold = Q3 + 1.5 * IQR

        windows = [w for i, w in enumerate(windows)
                   if motion_levels[i] <= threshold]

        # ---------- FEATURE EXTRACTION ----------
        for w in windows:

            try:
                # ===== HRV =====
                signals, info = nk.ppg_process(w["ppg"], sampling_rate=FS)
                peaks = info["PPG_Peaks"]

                if len(peaks) < 25:
                    continue

                hrv = nk.hrv_time(peaks, sampling_rate=FS)

                MeanNN = hrv["HRV_MeanNN"].values[0]
                SDNN = hrv["HRV_SDNN"].values[0]
                RMSSD = hrv["HRV_RMSSD"].values[0]

                # HRV sanity filter
                if RMSSD > 180:
                    continue
                if SDNN > 180:
                    continue

                # ===== EDA =====
                eda_signals, _ = nk.eda_process(w["gsr"], sampling_rate=FS)
                eda_feat = nk.eda_analyze(eda_signals, sampling_rate=FS)

                SCR_n = eda_feat["SCR_Peaks_N"].values[0]
                SCR_amp = eda_feat["SCR_Peaks_Amplitude_Mean"].values[0]
                Tonic_SD = eda_feat["EDA_Tonic_SD"].values[0]

                # EDA sanity filter (ta bort hela window)
                if SCR_n > 10:
                    continue
                if SCR_amp > 10:
                    continue
                if Tonic_SD > 50:
                    continue

                log_amp = np.log1p(SCR_amp)

                feature_rows.append({
                    "participant": participant_id,
                    "MeanNN_ms": MeanNN,
                    "SDNN_ms": SDNN,
                    "RMSSD_ms": RMSSD,
                    "SCR_Peaks_N": SCR_n,
                    "log_SCR_Amplitude": log_amp,
                    "EDA_Tonic_SD": Tonic_SD
                })

            except:
                continue

    except:
        continue

df_features = pd.DataFrame(feature_rows)

print("Totala windows kvar:", len(df_features))
print("Participants representerade:", df_features["participant"].nunique())

df_features.describe()


Processing files:  21%|██        | 281/1336 [03:15<09:10,  1.91it/s]c:\Users\pat19\OneDrive\Skrivbord\Thesis\SOM\som_env\Lib\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])
c:\Users\pat19\OneDrive\Skrivbord\Thesis\SOM\som_env\Lib\site-packages\neurokit2\eda\eda_intervalrelated.py:120: RuntimeWarning: Mean of empty slice
  output["SCR_Peaks_Amplitude_Mean"] = np.nanmean(data[peaks_idx]["SCR_Amplitude"].values)
Processing files: 100%|██████████| 1336/1336 [15:14<00:00,  1.46it/s]

Totala windows kvar: 3518
Participants representerade: 42


,participant,MeanNN_ms,SDNN_ms,RMSSD_ms,SCR_Peaks_N,log_SCR_Amplitude,EDA_Tonic_SD
count,3518.000000,3518.000000,3518.000000,3518.000000,3518.000000,3517.000000,3518.000000
mean,21.874360,765.090702,56.470260,60.213051,4.669983,1.123810,4.976944
std,11.724329,110.105456,26.167558,36.824078,1.758512,0.657638,6.498468
min,1.000000,427.734375,9.969024,7.851465,1.000000,0.005544,0.049783
25%,12.000000,685.267857,36.444269,32.974972,3.000000,0.552509,1.347385
50%,23.000000,761.410362,51.709706,48.660050,4.000000,1.134460,2.753651
75%,31.000000,838.953355,70.834095,80.156592,6.000000,1.670239,5.726767
max,42.000000,1143.437500,178.751372,179.567802,10.000000,2.396794,49.979011


In [102]:
df_features.describe()


,participant,MeanNN_ms,SDNN_ms,RMSSD_ms,SCR_Peaks_N,log_SCR_Amplitude,EDA_Tonic_SD
count,3518.000000,3518.000000,3518.000000,3518.000000,3518.000000,3517.000000,3518.000000
mean,21.874360,765.090702,56.470260,60.213051,4.669983,1.123810,4.976944
std,11.724329,110.105456,26.167558,36.824078,1.758512,0.657638,6.498468
min,1.000000,427.734375,9.969024,7.851465,1.000000,0.005544,0.049783
25%,12.000000,685.267857,36.444269,32.974972,3.000000,0.552509,1.347385
50%,23.000000,761.410362,51.709706,48.660050,4.000000,1.134460,2.753651
75%,31.000000,838.953355,70.834095,80.156592,6.000000,1.670239,5.726767
max,42.000000,1143.437500,178.751372,179.567802,10.000000,2.396794,49.979011


In [103]:
df_features = df_features.dropna()
df_features.to_csv("train_features.csv", index=False)
